In [ ]:
# imports

import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

if grok_api_key:
    print(f"Grok API Key exists and begins {grok_api_key[:4]}")
else:
    print("Grok API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:3]}")
else:
    print("OpenRouter API Key not set (and this is optional)")


In [ ]:
# Connect to OpenAI client library
# A thin wrapper around calls to HTTP endpoints

openai = OpenAI(api_key=openai_api_key)

# For Gemini, DeepSeek and Groq, we can use the OpenAI python client
# Because Google and DeepSeek have endpoints compatible with OpenAI
# And OpenAI allows you to change the base_url

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
deepseek_url = "https://api.deepseek.com"
groq_url = "https://api.groq.com/openai/v1"
grok_url = "https://api.x.ai/v1"
openrouter_url = "https://openrouter.ai/api/v1"
ollama_url = "http://localhost:11434/v1"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
deepseek = OpenAI(api_key=deepseek_api_key, base_url=deepseek_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
grok = OpenAI(api_key=grok_api_key, base_url=grok_url)
openrouter = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

## Going local

Just use the OpenAI library pointed to localhost:11434/v1

In [ ]:
requests.get("http://localhost:11434/").content

# If not running, run ollama serve at a command line

In [ ]:
!ollama pull qwen2.5:7b-instruct

# More advanced exercises

Try creating a 3-way, perhaps bringing Gemini into the conversation! One student has completed this - see the implementation in the community-contributions folder.

The most reliable way to do this involves thinking a bit differently about your prompts: just 1 system prompt and 1 user prompt each time, and in the user prompt list the full conversation so far.

Something like:

```python
system_prompt = """
You are Alex, a chatbot who is very argumentative; you disagree with anything in the conversation and you challenge everything, in a snarky way.
You are in a conversation with Blake and Charlie.
"""

user_prompt = f"""
You are Alex, in conversation with Blake and Charlie.
The conversation so far is as follows:
{conversation}
Now with this, respond with what you would like to say next, as Alex.
"""
```

Try doing this yourself before you look at the solutions. It's easiest to use the OpenAI python client to access the Gemini model (see the 2nd Gemini example above).

## Additional exercise

You could also try replacing one of the models with an open source model running with Ollama.

In [ ]:
participants = {
    "Alex" : {
        "client": openai,
        "model" : "gpt-4.1-mini",
        "system": """
        You are a chatbot who is very argumentative; you disagree with anything in the conversation and you challenge everything, in a snarky way.
        """
        
    },
    "Blake" : {
        "client": openrouter,
        "model" : "nvidia/nemotron-3-ultra-550b-a55b:free",
        "system": """
        You are a very polite, courteous chatbot. You try to agree with everything the other person says, or find common ground. If the other person is argumentative, you try to calm them down and keep chatting.
        """
    },

    "Charlie" : {
        "client": ollama,
        "model": "gemma3:270m",
        "system": """
        You are a fighter chatbot, and you are always right and ready to defend your position and point of view, you should always deny what the other person says and try to prove them wrong, you should always be aggressive and assertive in your responses.
        """
}
}

In [ ]:
def get_response(participant_name, messages):
    participant = participants[participant_name]
    system_message = {"role": "system", "content": participant["system"]}
    user_prompt = f"""You are {participant_name}, a chatbot participating in a conversation with other chatbots. You will respond to the messages in the conversation in a way that is consistent with your personality and role. You will not repeat what has already been said, and you will not ask questions that have already been answered. You will try to keep the conversation going and make it interesting for the other participants. You will not break character or reveal that you are a chatbot. You will not use any external information or knowledge that is not part of the conversation. You will not use any offensive or inappropriate language. You will not use any personal information or details about yourself or the other participants.
    The conversation so far is as follows:
    {messages}
Now you need to respond as {participant_name} in a way that is consistent with your personality and role. Your response should be concise, engaging, and relevant to the conversation.
"""
    prompts = [
        system_message,
        {"role": "user", "content": user_prompt}
    ]
    response = participant["client"].chat.completions.create(model=participant["model"], messages=prompts)
    return response.choices[0].message.content

In [ ]:
conversation = ["Hello, USA Market Got Crashed!"]
speaking_order = ["Alex", "Blake", "Charlie"]

for i in range(5):
    for participant in speaking_order:
        response = get_response(participant, conversation)
        conversation.append(f"{participant}: {response}")
        display(Markdown(f"### {participant}:\n{response}\n"))